# 04 — Evaluation and Error Analysis

## Metrics
All metrics are reported in **basis points** on the held-out test set
(last 20% of bars — never touched during training or validation).

| Metric | Description |
|--------|-------------|
| MAE    | Mean absolute error — primary metric, directly interpretable |
| RMSE   | Root mean squared error — emphasises large errors |
| MedAE  | Median absolute error — robust to outliers |

## Baselines
1. **Mean predictor** — always predicts training-set mean slippage
2. **Linear (Ridge)** — same features, linear model
3. **Heuristic** — β × order_size_fraction × vol_rolling, β calibrated on val

## Error analysis segments
Order size (Q1–Q4) · Volatility (Q1–Q4) · Time of day (open/mid/close) · Buy vs sell · Volume regime

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from features import FEATURE_NAMES
from pipeline import build_full_dataset
from model import SlippageMLP
from train import predict
from baselines import MeanPredictor, LinearBaseline, HeuristicBaseline
from evaluate import global_metrics, segment_breakdown
from paths import RESULTS_DIR
from viz import (
    plot_pred_vs_actual,
    plot_error_by_segment,
    plot_feature_importance,
    FIGURES_DIR,
)

In [ ]:
# Rebuild data and split via the shared pipeline. split.test_df is
# guaranteed to align row-by-row with split.y_test / X_test.
data, proxy_all, split = build_full_dataset()
n_tr, n_v, n_te = len(split.X_train), len(split.X_val), len(split.X_test)
print(f'Train: {n_tr:,}  Val: {n_v:,}  Test: {n_te:,}')

test_df = split.test_df  # no more iloc tricks

In [ ]:
# Load trained model
checkpoint = torch.load(RESULTS_DIR / 'model_checkpoint.pt', weights_only=True)
model = SlippageMLP(n_features=checkpoint['n_features'])
model.load_state_dict(checkpoint['state_dict'])
y_pred_mlp = predict(model, split.X_test)
y_test = split.y_test

In [ ]:
# Fit baselines
mean_pred = MeanPredictor().fit(split.X_train, split.y_train)
linear_pred = LinearBaseline().fit(split.X_train, split.y_train)
heuristic = HeuristicBaseline(FEATURE_NAMES)
heuristic.fit(split.X_val, split.y_val, split.scaler.mean_, split.scaler.scale_)

y_pred_mean = mean_pred.predict(split.X_test)
y_pred_linear = linear_pred.predict(split.X_test)
y_pred_heuristic = heuristic.predict(split.X_test, split.scaler.mean_, split.scaler.scale_)

In [ ]:
# Global metrics table
metrics = {
    'MLP': global_metrics(y_test, y_pred_mlp),
    'Linear': global_metrics(y_test, y_pred_linear),
    'Heuristic': global_metrics(y_test, y_pred_heuristic),
    'Mean': global_metrics(y_test, y_pred_mean),
}
metrics_df = pd.DataFrame(metrics).T
print('=== Global Metrics (test set, bps) ===')
print(metrics_df.round(2).to_string())

with open(RESULTS_DIR / 'metrics.json', 'w') as f:
    json.dump({k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in metrics.items()}, f, indent=2)

In [ ]:
# Predicted vs actual — coloured by vol regime
vol_q = pd.qcut(test_df['vol_rolling'], q=4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
fig = plot_pred_vs_actual(
    y_test, y_pred_mlp,
    regime=vol_q.values,
    regime_label='Vol quartile',
    title='MLP: Predicted vs Actual Slippage (bps)',
    save_as='pred_vs_actual.png',
)
plt.show()

In [ ]:
# Error breakdown by segment
breakdown = segment_breakdown(y_test, y_pred_mlp, test_df)
print(breakdown.to_string(index=False))
fig = plot_error_by_segment(breakdown, metric='mae_bps', save_as='error_by_segment.png')
plt.show()

In [ ]:
# Feature importance (permutation, on val set)
fig = plot_feature_importance(
    model, split.X_val, split.y_val, FEATURE_NAMES,
    n_repeats=5, seed=42, save_as='feature_importance.png',
)
plt.show()
print('All figures saved to results/figures/')